# 08 — Baseline leksykonowy (NRC) + testy istotności

Progresja: NRC bez ML → LogReg + TF-IDF char → HerBERT-base. Różnice testowane paired bootstrapem (95% CI na różnicy F1-Macro, resampling tego samego zbioru testowego).

In [1]:
import pickle, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from thesis_lib import optimal_thresholds, paired_bootstrap   # wspólne helpery (jeden plik do przeczytania)
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); plt.rcParams["savefig.dpi"]=300

EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RANDOM_STATE=42
PROCESSED_DIR=Path("../data/processed"); CACHE_DIR=Path("../data/features")
RESULTS_DIR=Path("../data/results"); FIGURES_DIR=Path("../figures")
CKPT=Path("../data/transformers/herbert-base-cased/checkpoint-14344")
device="cuda" if torch.cuda.is_available() else "cpu"

tw_val=pd.read_csv(PROCESSED_DIR/"twitteremo_val.csv"); tw_test=pd.read_csv(PROCESSED_DIR/"twitteremo_test.csv")
tw_train=pd.read_csv(PROCESSED_DIR/"twitteremo_train.csv")
for d in (tw_train,tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_train,y_val,y_test=(d[EMOTIONS].values for d in (tw_train,tw_val,tw_test))
TW=pickle.load(open(CACHE_DIR/"TW_FEATURES.pkl","rb"))

/path/to/repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Progi strojone na walidacji. 08 świadomie używa gęstszej siatki (0–0.95 co 0.005)
# niż reszta pracy (0.05–0.95 co 0.01) — zachowane przez jawne parametry, by liczby
# pozostały identyczne. Wspólna implementacja: thesis_lib.optimal_thresholds.
def opt_thresholds(yt, yp):
    return optimal_thresholds(yt, yp, lo=0.0, hi=0.95, step=0.005)

In [3]:
# --- Baseline 1: czysty leksykon NRC (bez ML) ---
# nrc kolumny = NRC_CATEGORIES: anger,anticipation,disgust,fear,joy,sadness,surprise,trust,positive,negative,coverage
# mapowanie na EMOTIONS:
NRC2EMO=[4,5,7,2,3,0,1,6]   # radość=joy, smutek=sadness, zaufanie=trust, wstręt=disgust, strach=fear, gniew=anger, przeczuwanie=anticipation, zdziwienie=surprise
nrc=TW["nrc"]
def nrc_scores(split): return nrc[split][:,NRC2EMO]
s_val, s_test = nrc_scores("val"), nrc_scores("test")
thr_lex=opt_thresholds(y_val,s_val)            # próg na udziale tokenów leksykonowych
pred_lex=(s_test>=thr_lex).astype(int)
print("NRC lexicon F1-Macro:",round(f1_score(y_test,pred_lex,average='macro',zero_division=0),3))

NRC lexicon F1-Macro: 0.243


In [4]:
# --- Klasyczny: LogReg + TF-IDF char ---
Xc=TW["tfidf_char"]
clf=OneVsRestClassifier(LogisticRegression(max_iter=1000,C=1.0,class_weight="balanced",
        solver="liblinear",random_state=RANDOM_STATE))
clf.fit(Xc["train"],y_train)
thr_c=opt_thresholds(y_val,clf.predict_proba(Xc["val"]))
pred_c=(clf.predict_proba(Xc["test"])>=thr_c).astype(int)
print("Classical F1-Macro:",round(f1_score(y_test,pred_c,average='macro',zero_division=0),3))

Classical F1-Macro: 0.472


In [5]:
# --- HerBERT-base z checkpointu ---
tok=AutoTokenizer.from_pretrained(CKPT)
model=AutoModelForSequenceClassification.from_pretrained(CKPT).to(device).eval()
@torch.no_grad()
def predict(texts,bs=64):
    out=[]
    for i in range(0,len(texts),bs):
        enc=tok(list(texts[i:i+bs]),truncation=True,max_length=128,padding=True,return_tensors="pt").to(device)
        out.append(expit(model(**enc).logits.float().cpu().numpy()))
    return np.vstack(out)
thr_h=opt_thresholds(y_val,predict(tw_val["tekst"].tolist()))
pred_h=(predict(tw_test["tekst"].tolist())>=thr_h).astype(int)
print("HerBERT F1-Macro:",round(f1_score(y_test,pred_h,average='macro',zero_division=0),3))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  36%|███▌      | 72/201 [00:00<00:00, 706.08it/s]

Loading weights:  71%|███████   | 143/201 [00:00<00:00, 563.79it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 596.93it/s]

HerBERT F1-Macro: 0.552


## PB-1: Progresja jakości (leksykon → ML → transformer)

In [6]:
systems={"NRC (leksykon, bez ML)":pred_lex,"LogReg + TF-IDF char":pred_c,"HerBERT-base":pred_h}
prog=pd.DataFrame([{"system":k,
    "f1_macro":round(f1_score(y_test,v,average='macro',zero_division=0),3),
    "f1_micro":round(f1_score(y_test,v,average='micro',zero_division=0),3)} for k,v in systems.items()])
prog.to_csv(RESULTS_DIR/"lexicon_progression.csv",index=False)
display(prog)

,system,f1_macro,f1_micro
0,"NRC (leksykon, bez ML)",0.243,0.325
1,LogReg + TF-IDF char,0.472,0.577
2,HerBERT-base,0.552,0.643


## PB-2: Paired bootstrap — istotność różnic

In [7]:
pairs=[("NRC (leksykon, bez ML)","LogReg + TF-IDF char"),
       ("LogReg + TF-IDF char","HerBERT-base"),
       ("NRC (leksykon, bez ML)","HerBERT-base")]
rows=[]
for a,b in pairs:
    md_,lo,hi,p=paired_bootstrap(y_test,systems[a],systems[b])
    sig="TAK" if (lo>0 or hi<0) else "nie"
    rows.append({"porównanie":f"{b} − {a}","Δ_f1_macro":round(md_,3),
                 "ci95":f"[{lo:+.3f}, {hi:+.3f}]","P(B>A)":round(p,3),"istotne_95":sig})
sig_df=pd.DataFrame(rows); sig_df.to_csv(RESULTS_DIR/"lexicon_significance.csv",index=False)
display(sig_df)

,porównanie,Δ_f1_macro,ci95,P(B>A),istotne_95
0,"LogReg + TF-IDF char − NRC (leksykon, bez ML)",0.228,"[+0.190, +0.265]",1.0,TAK
1,HerBERT-base − LogReg + TF-IDF char,0.080,"[+0.035, +0.125]",1.0,TAK
2,"HerBERT-base − NRC (leksykon, bez ML)",0.309,"[+0.266, +0.347]",1.0,TAK
